# Using Machine Learning techniques to predict oral cancer

## Imports

In [3]:
import kagglehub
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

ImportError: cannot import name '_promote' from 'scipy.spatial.transform._rotation' (c:\Users\Dell\anaconda3\envs\ml_env\Lib\site-packages\scipy\spatial\transform\_rotation.cp311-win_amd64.pyd)

## Load in data, basic data information

In [ ]:
path = kagglehub.dataset_download("fedesoriano/stroke-prediction-dataset")

file_path = os.path.join(path, "healthcare-dataset-stroke-data.csv")
df = pd.read_csv(file_path)

df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


## Data cleaning

In [ ]:
df.columns

Index(['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'work_type', 'Residence_type', 'avg_glucose_level', 'bmi',
       'smoking_status', 'stroke'],
      dtype='object')

### Checking data for unique values

Before modifying the data to be fit for training we should check how many unique values each feature can have, so that we can determine how to represent it (eg. use 0 and 1 for yes-no columns or one-hot encoding for those with more posiible values).

In [ ]:
for column in df.columns:
    print(f"{column} values: {df[column].unique()}")

id values: [ 9046 51676 31112 ... 19723 37544 44679]
gender values: ['Male' 'Female' 'Other']
age values: [6.70e+01 6.10e+01 8.00e+01 4.90e+01 7.90e+01 8.10e+01 7.40e+01 6.90e+01
 5.90e+01 7.80e+01 5.40e+01 5.00e+01 6.40e+01 7.50e+01 6.00e+01 5.70e+01
 7.10e+01 5.20e+01 8.20e+01 6.50e+01 5.80e+01 4.20e+01 4.80e+01 7.20e+01
 6.30e+01 7.60e+01 3.90e+01 7.70e+01 7.30e+01 5.60e+01 4.50e+01 7.00e+01
 6.60e+01 5.10e+01 4.30e+01 6.80e+01 4.70e+01 5.30e+01 3.80e+01 5.50e+01
 1.32e+00 4.60e+01 3.20e+01 1.40e+01 3.00e+00 8.00e+00 3.70e+01 4.00e+01
 3.50e+01 2.00e+01 4.40e+01 2.50e+01 2.70e+01 2.30e+01 1.70e+01 1.30e+01
 4.00e+00 1.60e+01 2.20e+01 3.00e+01 2.90e+01 1.10e+01 2.10e+01 1.80e+01
 3.30e+01 2.40e+01 3.40e+01 3.60e+01 6.40e-01 4.10e+01 8.80e-01 5.00e+00
 2.60e+01 3.10e+01 7.00e+00 1.20e+01 6.20e+01 2.00e+00 9.00e+00 1.50e+01
 2.80e+01 1.00e+01 1.80e+00 3.20e-01 1.08e+00 1.90e+01 6.00e+00 1.16e+00
 1.00e+00 1.40e+00 1.72e+00 2.40e-01 1.64e+00 1.56e+00 7.20e-01 1.88e+00
 1.24e+00 8.00e-01

### Adjusting column types

In [ ]:
df["age"] = df["age"].astype("uint8")
df

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
...,...,...,...,...,...,...,...,...,...,...,...,...
5105,18234,Female,80,1,0,Yes,Private,Urban,83.75,NaN,never smoked,0
5106,44873,Female,81,0,0,Yes,Self-employed,Urban,125.20,40.0,never smoked,0
5107,19723,Female,35,0,0,Yes,Self-employed,Rural,82.99,30.6,never smoked,0
5108,37544,Male,51,0,0,Yes,Private,Rural,166.29,25.6,formerly smoked,0


### Checking for NaN values

We check if there are any values missing from the dataset.
- `df.isna()` – returns a DataFrame of True/False values (True if the value in the original DataFrame is NaN, False otherwise)  
- `df.isna().any()` – for each column, checks if it contains at least one True value  
- `df.isna().any().any()` – returns True if there is **at least one NaN** in the original DataFrame, False otherwise

In [ ]:
df.isna().any().any()

True

### Dealing with NaNs

BMI is the only columns with missing values. We fill them using the gender average.

In [ ]:
df["bmi"] = df["bmi"].fillna(df.groupby("gender")["bmi"].transform("mean"))
df

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67,0,1,Yes,Private,Urban,228.69,36.600000,formerly smoked,1
1,51676,Female,61,0,0,Yes,Self-employed,Rural,202.21,29.065758,never smoked,1
2,31112,Male,80,0,1,Yes,Private,Rural,105.92,32.500000,never smoked,1
3,60182,Female,49,0,0,Yes,Private,Urban,171.23,34.400000,smokes,1
4,1665,Female,79,1,0,Yes,Self-employed,Rural,174.12,24.000000,never smoked,1
...,...,...,...,...,...,...,...,...,...,...,...,...
5105,18234,Female,80,1,0,Yes,Private,Urban,83.75,29.065758,never smoked,0
5106,44873,Female,81,0,0,Yes,Self-employed,Urban,125.20,40.000000,never smoked,0
5107,19723,Female,35,0,0,Yes,Self-employed,Rural,82.99,30.600000,never smoked,0
5108,37544,Male,51,0,0,Yes,Private,Rural,166.29,25.600000,formerly smoked,0


### Modifying the data

In [ ]:
# Binary to 1/0
df["ever_married"] = df["ever_married"].map({"Yes": 1, "No": 0})

# One-Hot Encoding
df = pd.get_dummies(df, columns=["gender", "work_type", "Residence_type", "smoking_status"])

# Convert all bool columns from OHE to int
bool_cols = df.select_dtypes(include="bool").columns
df[bool_cols] = df[bool_cols].astype(int)

df.head()

,id,age,hypertension,heart_disease,ever_married,avg_glucose_level,bmi,stroke,gender_Female,gender_Male,...,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0,9046,67,0,1,1,228.69,36.600000,1,0,1,...,0,1,0,0,0,1,0,1,0,0
1,51676,61,0,0,1,202.21,29.065758,1,1,0,...,0,0,1,0,1,0,0,0,1,0
2,31112,80,0,1,1,105.92,32.500000,1,0,1,...,0,1,0,0,1,0,0,0,1,0
3,60182,49,0,0,1,171.23,34.400000,1,1,0,...,0,1,0,0,0,1,0,0,0,1
4,1665,79,1,0,1,174.12,24.000000,1,1,0,...,0,0,1,0,1,0,0,0,1,0


In [ ]:
# Move label to the rightmost side of the dataframe
cols = [c for c in df.columns if c != "stroke"] + ["stroke"]
df = df[cols]

### Renaming the columns

In this step we standardize the naming convention of features in the dataframe.

In [ ]:
def get_renamed_column(name):
    name = name.strip()
    name = name.lower()
    name = name.replace(" ", "_")

    return name


def get_rename_dict(column_names):
    rename_dict = {}
    for name in column_names:
        rename_dict[name] = get_renamed_column(name)

    return rename_dict

In [ ]:
df = df.rename(columns=get_rename_dict(df.columns))
df.head()

,id,age,hypertension,heart_disease,ever_married,avg_glucose_level,bmi,gender_female,gender_male,gender_other,...,work_type_private,work_type_self-employed,work_type_children,residence_type_rural,residence_type_urban,smoking_status_unknown,smoking_status_formerly_smoked,smoking_status_never_smoked,smoking_status_smokes,stroke
0,9046,67,0,1,1,228.69,36.600000,0,1,0,...,1,0,0,0,1,0,1,0,0,1
1,51676,61,0,0,1,202.21,29.065758,1,0,0,...,0,1,0,1,0,0,0,1,0,1
2,31112,80,0,1,1,105.92,32.500000,0,1,0,...,1,0,0,1,0,0,0,1,0,1
3,60182,49,0,0,1,171.23,34.400000,1,0,0,...,1,0,0,0,1,0,0,0,1,1
4,1665,79,1,0,1,174.12,24.000000,1,0,0,...,0,1,0,1,0,0,0,1,0,1


### Check for duplicates

In [ ]:
any(df.duplicated())

False

There are only unique records in the dataset already, we may proceed.

### Check for class imbalance

There should be a similar number of records in each class.

In [ ]:
df[df.columns[-1]].sum() / df[df.columns[-1]].size

0.0487279843444227

## Preparing the train and test sets

### Train-test split

In [ ]:
columns_to_drop = []
df = df.drop(columns=columns_to_drop)

In [ ]:
X = df.drop(columns=[df.columns[-1]]) 
y = df[df.columns[-1]]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Feature scaling

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Create and train models

### Random Forests

In [ ]:
random_forest_models = {
    "RandomForest_100_5": RandomForestClassifier(
        n_estimators=100,
        max_depth=5
    ),
    "RandomForest_50_5": RandomForestClassifier(
        n_estimators=50,
        max_depth=5
    ),
    "RandomForest_50_10": RandomForestClassifier(
        n_estimators=50,
        max_depth=10
    ),
    "RandomForest_500_3": RandomForestClassifier(
        n_estimators=500,
        max_depth=3
    ),
    "RandomForest_200_10": RandomForestClassifier(
        n_estimators=200,
        max_depth=10
    ),
    "RandomForest_300_7": RandomForestClassifier(
        n_estimators=300,
        max_depth=7
    ),
}

In [ ]:
random_forest_scores = {}
for name, model in random_forest_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)

    scores = {"Accuracy": accuracy, "Precision": precision, "Recall": recall}
    random_forest_scores[name] = scores

    print(f"{name} Accuracy: {accuracy}")
    print(f"{name} Precision: {precision}")
    print(f"{name} Recall: {recall}\n")

print(random_forest_scores)

c:\Users\Dell\anaconda3\envs\ml_env\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  jaccard_score : Compute the Jaccard similarity coefficient score.


RandomForest_100_5 Accuracy: 0.9393346379647749
RandomForest_100_5 Precision: 0.0
RandomForest_100_5 Recall: 0.0



c:\Users\Dell\anaconda3\envs\ml_env\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  jaccard_score : Compute the Jaccard similarity coefficient score.


RandomForest_50_5 Accuracy: 0.9393346379647749
RandomForest_50_5 Precision: 0.0
RandomForest_50_5 Recall: 0.0



c:\Users\Dell\anaconda3\envs\ml_env\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  jaccard_score : Compute the Jaccard similarity coefficient score.


...

c:\Users\Dell\anaconda3\envs\ml_env\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  jaccard_score : Compute the Jaccard similarity coefficient score.


RandomForest_500_3 Accuracy: 0.9393346379647749
RandomForest_500_3 Precision: 0.0
RandomForest_500_3 Recall: 0.0



c:\Users\Dell\anaconda3\envs\ml_env\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  jaccard_score : Compute the Jaccard similarity coefficient score.


RandomForest_200_10 Accuracy: 0.9393346379647749
RandomForest_200_10 Precision: 0.0
RandomForest_200_10 Recall: 0.0

RandomForest_300_7 Accuracy: 0.9393346379647749
RandomForest_300_7 Precision: 0.0
RandomForest_300_7 Recall: 0.0

{'RandomForest_100_5': {'Accuracy': 0.9393346379647749, 'Precision': 0.0, 'Recall': 0.0}, 'RandomForest_50_5': {'Accuracy': 0.9393346379647749, 'Precision': 0.0, 'Recall': 0.0}, 'RandomForest_50_10': {'Accuracy': 0.9393346379647749, 'Precision': 0.0, 'Recall': 0.0}, 'RandomForest_500_3': {'Accuracy': 0.9393346379647749, 'Precision': 0.0, 'Recall': 0.0}, 'RandomForest_200_10': {'Accuracy': 0.9393346379647749, 'Precision': 0.0, 'Recall': 0.0}, 'RandomForest_300_7': {'Accuracy': 0.9393346379647749, 'Precision': 0.0, 'Recall': 0.0}}


c:\Users\Dell\anaconda3\envs\ml_env\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  jaccard_score : Compute the Jaccard similarity coefficient score.
